# Borusyak–Jaravel–Spiess Imputation DiD

**Econometrics Notebook Library · pedagogical reference notebook**

This notebook is deliberately written to **show the estimator rather than hide it inside a helper function**. The production helper in `econnotes.panel` appears only near the end as a numerical cross-check.

The core reference is Borusyak, Jaravel & Spiess, *Revisiting Event-Study Designs: Robust and Efficient Estimation* (Review of Economic Studies, 2024). The authors' `did_imputation` implementation describes the estimator as three separate steps: estimate the untreated outcome model on untreated observations, impute untreated potential outcomes for treated observations, and aggregate the resulting observation-level treatment-effect estimates.

## 1. What is being estimated?

Let treatment be absorbing and let $D_{it}=1$ once unit $i$ has adopted treatment. Under the benchmark untreated-outcome model,

$$
Y_{it}(0)=\alpha_i+\lambda_t+\varepsilon_{it}.
$$

The key restriction is imposed on **untreated potential outcomes**, not on the path of treatment effects. For a treated observation we want

$$
\tau_{it}=Y_{it}(1)-Y_{it}(0).
$$

BJS recover it by estimating $Y_{it}(0)$ without using treated outcomes, extrapolating that model to treated observations, and then computing

$$
\widehat\tau_{it}=Y_{it}-\widehat Y_{it}(0).
$$

An event-time estimand is an explicit average of those observation-level effects, for example

$$
\widehat\tau_h = \frac{1}{N_h}\sum_{i,t:\,t-E_i=h}\widehat\tau_{it}.
$$

That ordering matters: **counterfactual first, treatment-effect aggregation second**.

## 2. Simulate a staggered-adoption panel

The simulation has heterogeneous and dynamic treatment effects. Untreated outcomes satisfy the additive unit/time fixed-effect structure used below. The time-invariant covariate `x` is absorbed by unit fixed effects in this benchmark, so it is intentionally omitted from the visible untreated-outcome regression.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

from econnotes.panel import simulate_staggered_panel

plt.rcParams["figure.figsize"] = (8, 4.5)
pd.set_option("display.max_columns", 30)

df = simulate_staggered_panel(n_units=260, seed=51)
df.head()

In [ ]:
cohort_summary = (
    df[["unit", "cohort"]]
    .drop_duplicates()
    .assign(cohort_label=lambda x: x["cohort"].replace(np.inf, "never"))
    .groupby("cohort_label", dropna=False)
    .size()
    .rename("units")
)
cohort_summary

## 3. Before estimating anything: check imputation support

Imputation is only meaningful for treated observations whose unit and calendar-time fixed effects can be learned from untreated data. In this simple panel, that means:

- each treated unit must contribute untreated observations before adoption; and
- each calendar period containing treated observations must still contain untreated observations.

The authors' production command performs more general checks because richer fixed effects and controls can create additional non-imputability.

In [ ]:
untreated = df["treated"].eq(0)
treated = df["treated"].eq(1)

treated_units = set(df.loc[treated, "unit"])
units_seen_untreated = set(df.loc[untreated, "unit"])
treated_times = set(df.loc[treated, "time"])
times_seen_untreated = set(df.loc[untreated, "time"])

missing_unit_support = sorted(treated_units - units_seen_untreated)
missing_time_support = sorted(treated_times - times_seen_untreated)

support_check = pd.Series({
    "treated units without untreated history": len(missing_unit_support),
    "treated calendar times without untreated observations": len(missing_time_support),
    "untreated observations used in step 1": int(untreated.sum()),
    "treated observations to impute": int(treated.sum()),
})

assert not missing_unit_support
assert not missing_time_support
support_check

## 4. Step 1 — estimate the untreated outcome model **using untreated observations only**

This is the part that should be visible in a teaching notebook. We estimate

$$
Y_{it}=\alpha_i+\lambda_t+\varepsilon_{it}
$$

only where $D_{it}=0$. Treated outcomes do not help fit the counterfactual model.

In [ ]:
fit_y0 = smf.ols(
    "y ~ C(unit) + C(time)",
    data=df.loc[untreated],
).fit()

step1_summary = pd.Series({
    "n untreated observations": int(fit_y0.nobs),
    "R-squared on untreated sample": float(fit_y0.rsquared),
})
step1_summary

## 5. Step 2 — extrapolate $\widehat Y_{it}(0)$ to treated observations

Now use the fixed effects learned in Step 1 to predict what each observation would have looked like without treatment. For treated observations, subtract that imputed counterfactual from the observed outcome.

In [ ]:
work = df.copy()
work["y0_hat"] = fit_y0.predict(work)
work["tau_hat"] = np.where(
    work["treated"].eq(1),
    work["y"] - work["y0_hat"],
    np.nan,
)

work.loc[work["treated"].eq(1), [
    "unit", "time", "cohort", "event_time", "y", "y0_hat", "tau_hat"
]].head(10)

## 6. Step 3 — aggregate the imputed observation-level effects

Nothing mysterious remains. Here the target is the equal-weight event-time ATT among treated observations observed at horizon $h$. Different scientific questions can use different prespecified weights.

In [ ]:
bjs_manual = (
    work.loc[work["treated"].eq(1)]
    .groupby("event_time", as_index=False)
    .agg(
        estimate=("tau_hat", "mean"),
        n=("tau_hat", "size"),
    )
    .sort_values("event_time")
)

truth = (
    df.loc[df["treated"].eq(1)]
    .groupby("event_time", as_index=False)["tau_true"]
    .mean()
    .rename(columns={"tau_true": "truth"})
)

plot = bjs_manual.merge(truth, on="event_time", how="left")
plot

In [ ]:
fig, ax = plt.subplots()
ax.plot(plot["event_time"], plot["truth"], marker="o", label="Truth")
ax.plot(plot["event_time"], plot["estimate"], marker="o", label="Manual BJS imputation")
ax.axhline(0, linewidth=1)
ax.set(
    xlabel="Event time",
    ylabel="Effect",
    title="BJS mechanics shown explicitly: impute first, aggregate second",
)
ax.legend();

### A note on uncertainty

The graph above intentionally does **not** attach the naive standard error of the imputed residuals. Those residuals share estimated fixed effects and can be dependent within unit, so `sd(tau_hat)/sqrt(n)` is not the BJS publication-grade variance estimator. The purpose here is to expose the point estimator. For empirical work, use the authors' production implementation (or another implementation validated against it) for the appropriate covariance estimator.

## 7. Validate the transparent calculation against the library helper

Only now do we call the repository's compact helper. It is a regression test, not the explanation of the estimator.

In [ ]:
from econnotes.panel import bjs_imputation

helper_result, _ = bjs_imputation(df)
comparison = bjs_manual.merge(
    helper_result[["event_time", "estimate"]].rename(columns={"estimate": "helper_estimate"}),
    on="event_time",
    how="left",
)
comparison["abs_diff"] = (comparison["estimate"] - comparison["helper_estimate"]).abs()

assert comparison["abs_diff"].max() < 1e-10
comparison

# Diagnostics are a separate exercise

The original version of this notebook blurred together one residual diagnostic and the BJS identification checks. They should be separated.

The authors' `did_imputation` documentation is explicit: a pre-trend test is a **separate regression**, performed on untreated observations only. With `pretrends(K)`, the untreated outcome is regressed on the same fixed effects/controls plus indicators for the $K$ periods immediately before treatment. The lead coefficients are then tested jointly.

This does **not** change the post-treatment point estimates above. It asks whether the untreated-outcome model is contradicted by systematic deviations just before treatment.

## 8. BJS-style pre-trend regression and joint Wald test

For $K=3$, construct indicators for event times $-1,-2,-3$ among untreated observations and estimate

$$
Y_{it}=\alpha_i+\lambda_t+\gamma_1 1[h=-1]+\gamma_2 1[h=-2]+\gamma_3 1[h=-3]+u_{it}
$$

on the untreated sample only. The null is

$$
H_0:\gamma_1=\gamma_2=\gamma_3=0.
$$

As in the authors' command, the reference category is the remaining untreated observations: earlier pre-treatment periods together with never-treated observations.

In [ ]:
K = 3
diag = df.copy()
for j in range(1, K + 1):
    diag[f"pre{j}"] = (diag["event_time"].eq(-j) & diag["treated"].eq(0)).astype(int)

pre_terms = " + ".join(f"pre{j}" for j in range(1, K + 1))
pre_fit = smf.ols(
    f"y ~ C(unit) + C(time) + {pre_terms}",
    data=diag.loc[diag["treated"].eq(0)],
).fit(
    cov_type="cluster",
    cov_kwds={"groups": diag.loc[diag["treated"].eq(0), "unit"]},
)

pre_rows = []
for j in range(1, K + 1):
    name = f"pre{j}"
    pre_rows.append({
        "event_time": -j,
        "estimate": float(pre_fit.params[name]),
        "se_cluster": float(pre_fit.bse[name]),
    })
pretrend = pd.DataFrame(pre_rows).sort_values("event_time")

restrictions = ", ".join(f"pre{j} = 0" for j in range(1, K + 1))
joint_test = pre_fit.wald_test(restrictions, scalar=True)

pretrend, {
    "joint Wald statistic": float(joint_test.statistic),
    "joint p-value": float(joint_test.pvalue),
}

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(
    pretrend["event_time"],
    pretrend["estimate"],
    yerr=1.96 * pretrend["se_cluster"],
    marker="o",
    capsize=3,
)
ax.axhline(0, linewidth=1)
ax.set(
    xlabel="Event time",
    ylabel="Pre-trend coefficient",
    title=f"Separate BJS pre-trend regression: joint p = {float(joint_test.pvalue):.3f}",
);

### How to read this

Failing to reject the joint null is **not proof** of parallel trends; pre-trend tests can have low power. Rejecting it is evidence against the maintained untreated-outcome model/no-anticipation restrictions. The number of leads should be chosen reasonably rather than mechanically using every available pre-period; the authors' implementation warns that an overly large set can reduce power.

## 9. Residual visualization — useful, but not *the* BJS diagnostic

The earlier notebook reported an untreated-residual diagnostic without distinguishing its provenance. Following Kirill Borusyak's feedback, treat this as a **Liu, Wang and Xu (2022)-style counterfactual diagnostic/visualization**, not as the unique BJS test.

Below we inspect mean untreated residuals by relative time among eventually treated units. It is a descriptive way to ask whether the fitted untreated counterfactual develops systematic prediction errors as treatment approaches. It complements rather than replaces the separate BJS pre-trend regression above.

In [ ]:
pre_resid = work.loc[
    work["treated"].eq(0) & np.isfinite(work["cohort"])
].copy()
pre_resid["resid0"] = pre_resid["y"] - pre_resid["y0_hat"]

resid_by_event = (
    pre_resid
    .groupby("event_time", as_index=False)
    .agg(
        mean_resid=("resid0", "mean"),
        sd_resid=("resid0", "std"),
        n=("resid0", "size"),
    )
    .sort_values("event_time")
)
resid_by_event

In [ ]:
fig, ax = plt.subplots()
ax.plot(
    resid_by_event["event_time"],
    resid_by_event["mean_resid"],
    marker="o",
)
ax.axhline(0, linewidth=1)
ax.set(
    xlabel="Event time (untreated observations only)",
    ylabel="Mean untreated residual",
    title="Residual diagnostic shown separately from the BJS pre-trend test",
);

## 10. Other checks and failure modes

There is no single diagnostic that validates an event-study design. At minimum, separate these questions:

1. **Can the target effects be imputed?** Audit support before estimation; do not silently change the estimand when some treated observations cannot be imputed.
2. **Does the untreated-outcome model show pre-treatment violations?** Use a separate lead regression/joint test such as the BJS `pretrends(K)` construction above.
3. **Do residual/counterfactual diagnostics reveal systematic misspecification?** Residual visualizations can expose patterns a single joint test may obscure.
4. **Would placebo timing generate apparent effects?** The authors' implementation permits shifted treatment timing as a placebo exercise, while explicitly recommending the formal pre-trend test for the main parallel-trends check.
5. **Are the target weights scientifically meaningful?** BJS allow arbitrary heterogeneous treatment effects, but the reported average still depends on which treated observations and horizons enter the estimand.

## 11. Why imputation can still fail

Imputation is not magic matrix completion. If untreated potential outcomes do not satisfy the posited fixed-effect/parallel-trends structure, $\widehat Y_{it}(0)$ can be systematically wrong even when the fitted untreated regression looks precise.

Identification comes from the untreated-outcome restrictions and support, not from prediction accuracy alone. Pre-trend diagnostics probe those restrictions; they do not prove them.

## Researcher checklist

- Fit the untreated outcome model on **untreated observations only**.
- Show the imputation mechanics explicitly when the notebook is pedagogical.
- Verify unit/time support before extrapolating to treated observations.
- Define the target treated observations/horizons and their weights before aggregation.
- Keep pre-trend testing separate from post-treatment effect estimation.
- Label residual diagnostics by their actual provenance; do not present one visualization as the only available test.
- Do not use naive residual standard errors as publication-grade BJS inference.
- Validate teaching code against a production implementation.

### References

- Borusyak, K., Jaravel, X., & Spiess, J. (2024). *Revisiting Event-Study Designs: Robust and Efficient Estimation*. Review of Economic Studies. https://doi.org/10.1093/restud/rdae007
- Authors' Stata implementation and help file: https://github.com/borusyak/did_imputation
- Liu, L., Wang, Y., & Xu, Y. (2022), counterfactual-estimator diagnostic framework (residual diagnostic cited here as a distinct approach, not as the BJS pre-trend test).